# Models v2: LSTM and Prophet vs baselines

Thin wrapper around `src/lstm.py`, `src/prophet_model.py` and `src/compare.py`. Full training takes ~10 min on 8 CPU cores, so it is off by default; the run writes `predictions_lstm.parquet` and `predictions_prophet.parquet` to `processed/`.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import pandas as pd
from src.data_loader import data_dir, load_features
from src.lstm import run_lstm
from src.prophet_model import run_prophet
from src import compare

PROCESSED = data_dir() / "processed"
RETRAIN = False

In [2]:
if RETRAIN:
    df = load_features()
    run_lstm(df).to_parquet(PROCESSED / "predictions_lstm.parquet", index=False)
    run_prophet(df).to_parquet(PROCESSED / "predictions_prophet.parquet", index=False)

Metrics are computed only on test rows that all four models predict (written to `predictions_v2.csv` / `metrics_v2.csv`).

In [3]:
metrics, metrics_no_prophet = compare.main()
metrics.pivot_table(index=["subset", "horizon_h"], columns="model", values="MAE").round(2)

model                         hist_gradient_boosting   lstm  persistence  \
subset             horizon_h                                               
test_all           24                          33.34  32.82        36.15   
                   48                          43.87  42.18        47.06   
test_excl_lockdown 24                          35.68  35.43        39.42   
                   48                          47.38  46.08        51.98   

model                         prophet  
subset             horizon_h           
test_all           24           39.24  
                   48           49.84  
test_excl_lockdown 24           39.31  
                   48           49.96

In [4]:
metrics.pivot_table(index=["subset", "horizon_h"], columns="model", values="RMSE").round(2)

model                         hist_gradient_boosting   lstm  persistence  \
subset             horizon_h                                               
test_all           24                          55.18  55.77        62.96   
                   48                          68.92  68.60        78.80   
test_excl_lockdown 24                          59.67  60.49        68.29   
                   48                          74.70  74.56        86.01   

model                         prophet  
subset             horizon_h           
test_all           24           61.26  
                   48           74.71  
test_excl_lockdown 24           63.35  
                   48           76.62

Rows compared in v2 vs all target-valid test rows in v1.

In [5]:
v1 = pd.read_csv(PROCESSED / "metrics_v1.csv")
key = ["subset", "horizon_h"]

def n_rows(table, name):
    return table[table["model"] == "persistence"].set_index(key)["n_rows"].rename(name)

rows = pd.concat([n_rows(v1, "v1_test_rows"), n_rows(metrics, "v2_all_four"), n_rows(metrics_no_prophet, "v2_excl_prophet")], axis=1)
rows

,,v1_test_rows,v2_all_four,v2_excl_prophet
subset,horizon_h,,,
test_all,24,841299,618906,788298
test_excl_lockdown,24,628304,475735,589057
test_all,48,830031,611632,778536
test_excl_lockdown,48,621820,471426,583658


Persistence, HGB and LSTM on their own intersection (Prophet excluded, so the 31 stations without train rows stay in).

In [6]:
metrics_no_prophet.pivot_table(index=["subset", "horizon_h"], columns="model", values=["MAE", "RMSE"]).round(2)

MAE                     \
model                        hist_gradient_boosting   lstm persistence   
subset             horizon_h                                             
test_all           24                         31.38  30.90       33.42   
                   48                         41.42  39.55       43.61   
test_excl_lockdown 24                         34.12  33.96       37.17   
                   48                         45.50  44.04       49.15   

                                               RMSE                     
model                        hist_gradient_boosting   lstm persistence  
subset             horizon_h                                            
test_all           24                         51.65  52.27       58.65  
                   48                         64.82  64.32       73.55  
test_excl_lockdown 24                         56.37  57.28       64.23  
                   48                         71.03  70.75       81.11

## Verdict (MAE, all-four intersection; gaps under 1.0 count as a tie)

**test_all:** LSTM ties HGB at 24h (32.82 vs 33.34) and beats it at 48h (42.18 vs 43.87), beats persistence at both horizons (36.15, 47.06), and its 24h RMSE is slightly worse than HGB's (55.77 vs 55.18); flat-growth Prophet loses to persistence and HGB at both horizons (39.24, 49.84).

**test_excl_lockdown:** LSTM ties HGB at 24h (35.43 vs 35.68) and beats it at 48h (46.08 vs 47.38), beats persistence at both horizons (39.42, 51.98), and its 24h RMSE is again slightly worse than HGB's (60.49 vs 59.67); Prophet ties persistence at 24h (39.31 vs 39.42), beats it at 48h (49.96 vs 51.98), and loses to HGB at both.